# 03. 데이터 전처리 — 시험 점수의 절반이 여기서 나와요

> **배우는 것**: 이상치 제거(조건 / IQR) · 결측치 확인·삭제·채우기 · 컬럼 삭제 · 문자열→숫자 · `LabelEncoder` · `map` · `astype` · `get_dummies` · `train_test_split` · `MinMaxScaler` / `StandardScaler`
> ⏱ 예상 소요시간: 70분  |  📑 같이 볼 슬라이드: `slides/03_데이터_전처리.html`

**왜 전처리?** 머신러닝 모델은 **숫자만** 먹고, **빈칸(NaN)** 을 싫어하고, **튀는 값**에 휘둘려요.
요리로 치면 재료 손질 단계예요. 손질이 끝나야 요리(모델링)를 할 수 있어요.

공식 샘플 문항 기준 전처리 흐름 (변수명까지 그대로 나와요):
```
df  →(이상치 제거) df_temp  →(결측 제거) df_na  →(컬럼 삭제) df_del  →(원-핫) df_preset  →(분할) X_train, X_valid, y_train, y_valid
```
이 노트북에서는 단계마다 `df` 를 계속 갱신하며 연습합니다.

In [ ]:
# ▶ 맨 처음 한 번 실행하세요 (데이터 경로 + 자동 채점 도구 준비)
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'          # 데이터 폴더 위치
import aice_check as check     # check.p01_q1() 처럼 쓰는 채점 도구

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_DIR + 'telco_churn.csv')     # 통신사 고객 이탈
house = pd.read_csv(DATA_DIR + 'house_price.csv')  # 아파트 가격 (IQR, fillna 연습용)
df.shape, house.shape

## 1. 쓸모없는 컬럼 지우기

### ✏️ 문제 1. 컬럼 삭제
고객 번호 `customerID` 는 예측에 도움이 안 되는 **식별자**예요. 삭제하고 결과를 다시 **`df`** 에 저장하세요.

In [ ]:
df = df.___('customerID', axis=___)
df.columns

<details><summary>💡 힌트 (막히면 클릭)</summary>

`df = df.drop('컬럼', axis=1)` — 결과를 다시 df 에 넣어야 실제로 바뀌어요!

</details>

In [ ]:
check.p03_q1()

## 2. 이상치(outlier) 제거
### 방법 ① 기준값으로 자르기 — 시험에 가장 많이 나오는 형태
문제에서 "Speed_Per_Hour 가 **300 이상**인 데이터를 삭제하라" 처럼 기준을 줘요.
**삭제할 조건의 반대**를 남기면 됩니다: `df = df[df['컬럼'] < 300]`

### ✏️ 문제 2. 기준값으로 이상치 제거
`MonthlyCharges` 가 **200 이상**인 행을 제거한 결과를 다시 **`df`** 에 저장하세요.

In [ ]:
df = df[df['MonthlyCharges'] ___]
df['MonthlyCharges'].max(), len(df)

<details><summary>💡 힌트 (막히면 클릭)</summary>

'200 이상 삭제' = '200 미만만 남기기' → `< 200`

</details>

In [ ]:
check.p03_q2()

### 방법 ② IQR 규칙 — 기준을 안 줄 때 쓰는 통계적 방법
- Q1 = 하위 25% 값, Q3 = 하위 75% 값, **IQR = Q3 − Q1**
- 정상 범위: **Q1 − 1.5×IQR ~ Q3 + 1.5×IQR** (boxplot 의 수염 끝과 같아요)

### ✏️ 문제 3. IQR 로 이상치 제거
`house` 의 `area_m2`(면적) 로 IQR 정상 범위를 계산하고, 범위 안의 행만 **`house_clean`** 에 저장하세요.

In [ ]:
q1 = house['area_m2'].___
q3 = house['area_m2'].___
iqr = ___
lower = q1 - ___ * iqr
upper = q3 + 1.5 * iqr
house_clean = house[(house['area_m2'] >= lower) ___ (house['area_m2'] <= upper)]
print(f'정상 범위: {lower:.1f} ~ {upper:.1f}, 남은 행: {len(house_clean)} / {len(house)}')

<details><summary>💡 힌트 (막히면 클릭)</summary>

`quantile(0.25)`, `quantile(0.75)`, 조건 두 개는 `&` 로

</details>

In [ ]:
check.p03_q3()

## 3. 결측치(NaN) 처리
| 상황 | 방법 | 코드 |
|---|---|---|
| 결측이 **아주 많은 컬럼** (절반 가까이) | 컬럼째 삭제 | `df = df.drop('컬럼', axis=1)` |
| 결측이 **조금** 있는 행 | 행 삭제 | `df = df.dropna()` |
| 숫자 컬럼, 행을 지우기 아까울 때 | 평균/중앙값으로 채우기 | `fillna(df['a'].median())` |
| 문자 컬럼 | 최빈값으로 채우기 | `fillna(df['a'].mode()[0])` |

> 📝 시험에서는 "결측치가 있는 **행을 삭제**하여 `df_na` 에 저장" 처럼 방법을 정해줘요. 문제를 꼼꼼히!

### ✏️ 문제 4. 결측치 확인
컬럼별 결측 개수를 **`null_counts`** 에 저장하세요.

In [ ]:
null_counts = df.___
null_counts[null_counts > 0]

<details><summary>💡 힌트 (막히면 클릭)</summary>

`df.isnull().sum()` 또는 `df.isna().sum()`

</details>

In [ ]:
check.p03_q4()

### ✏️ 문제 5. 결측 많은 컬럼 삭제
결측이 절반 가까이 되는 `DeviceProtection` 컬럼을 삭제해 **`df`** 에 저장하세요.

In [ ]:
df = df.drop(___, axis=1)

<details><summary>💡 힌트 (막히면 클릭)</summary>

여러 개를 지울 땐 리스트: `drop(['a', 'b'], axis=1)`

</details>

In [ ]:
check.p03_q5()

### ✏️ 문제 6. 결측 행 삭제
남은 결측치가 있는 **행**을 모두 삭제해 **`df`** 에 저장하세요.

In [ ]:
df = df.___
df.isnull().sum().sum(), len(df)

<details><summary>💡 힌트 (막히면 클릭)</summary>

`dropna()` 의 기본은 axis=0 (행 삭제). 공식 샘플: `df_na = df_temp.dropna(axis=0)`

</details>

In [ ]:
check.p03_q6()

### ✏️ 문제 7. 숫자 결측 채우기
`house` 의 `built_year` 는 **중앙값(median)**, `station_dist_m` 는 **평균(mean)** 으로 결측을 채워 각 컬럼에 다시 저장하세요.

In [ ]:
house['built_year'] = house['built_year'].___(house['built_year'].___)
house['station_dist_m'] = house['station_dist_m'].fillna(house['station_dist_m'].___)
house.isnull().sum()

<details><summary>💡 힌트 (막히면 클릭)</summary>

평균은 이상치에 끌려가고, 중앙값은 덜 끌려가요. 치우친 데이터면 median 이 안전해요.

</details>

In [ ]:
check.p03_q7()

### ✏️ 문제 8. 문자 결측 채우기 (최빈값)
새로 읽은 `tmp` 의 `gender` 결측을 **최빈값**으로 채우세요. 최빈값은 **`gender_mode`** 에 저장합니다.

In [ ]:
tmp = pd.read_csv(DATA_DIR + 'telco_churn.csv')
gender_mode = tmp['gender'].___
tmp['gender'] = tmp['gender'].fillna(___)
gender_mode, tmp['gender'].isnull().sum()

<details><summary>💡 힌트 (막히면 클릭)</summary>

`mode()` 는 최빈값이 여러 개일 수 있어서 Series 로 돌려줘요 → `[0]` 으로 첫 번째 값

</details>

In [ ]:
check.p03_q8()

## 4. 숫자로 바꾸기 (인코딩)
모델은 `'Yes'`, `'Female'` 같은 **문자열을 계산할 수 없어요.** 모두 숫자로 바꿔야 합니다.

### 4-1. 숫자처럼 생긴 문자열 → `pd.to_numeric`
`TotalCharges` 에는 `' '`(공백)가 섞여 있어서 `astype(float)` 이 실패해요.
`errors='coerce'` 를 주면 **숫자로 못 바꾸는 값은 NaN** 으로 바꿔줘요 → 그다음 `fillna`.

### ✏️ 문제 9. ' ' 공백 → 숫자
`TotalCharges` 를 숫자로 바꾸고, 생긴 NaN 은 `0` 으로 채워 다시 저장하세요.

In [ ]:
df['TotalCharges'] = pd.___(df['TotalCharges'], errors=___)
df['TotalCharges'] = df['TotalCharges'].___
df['TotalCharges'].dtype

<details><summary>💡 힌트 (막히면 클릭)</summary>

tenure 가 0 인 신규 고객이라 누적요금 0 이 맞아요.

</details>

In [ ]:
check.p03_q9()

### 4-2. 타깃(정답) 컬럼 → `LabelEncoder`
글자를 **알파벳 순서대로 0, 1, 2…** 로 바꿔줘요. (`No` → 0, `Yes` → 1)
```python
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['컬럼'] = le.fit_transform(df['컬럼'])
```

### ✏️ 문제 10. LabelEncoder
`Churn` 을 LabelEncoder 로 0/1 로 바꾸세요. 인코더는 **`le`** 에 저장합니다.

In [ ]:
from sklearn.preprocessing import ___
le = ___
df['Churn'] = le.___(df['Churn'])
le.classes_, df['Churn'].value_counts()

<details><summary>💡 힌트 (막히면 클릭)</summary>

`le.classes_` 의 순서가 곧 숫자 번호예요: ['No', 'Yes'] → No=0, Yes=1

</details>

In [ ]:
check.p03_q10()

### ✏️ 문제 11. map 으로 직접 지정
`gender` 를 `'Male'` → 1, `'Female'` → 0 으로 바꾸세요.

In [ ]:
df['gender'] = df['gender'].map(___)
df['gender'].value_counts()

<details><summary>💡 힌트 (막히면 클릭)</summary>

직접 번호를 정하고 싶으면 map / replace, 자동으로 매기려면 LabelEncoder

</details>

In [ ]:
check.p03_q11()

### ✏️ 문제 12. 타입 변환
`SeniorCitizen` 은 0.0/1.0 실수예요. **정수(int)** 로 바꾸세요.

In [ ]:
df['SeniorCitizen'] = df['SeniorCitizen'].___
df['SeniorCitizen'].dtype

<details><summary>💡 힌트 (막히면 클릭)</summary>

결측이 남아 있으면 int 변환이 실패해요 (그래서 dropna 를 먼저 했어요).

</details>

In [ ]:
check.p03_q12()

### 4-3. 나머지 범주형 → 원-핫 인코딩 `pd.get_dummies`
`Contract` 가 3종류면 → `Contract_Month-to-month`, `Contract_One year`, `Contract_Two year` **3개 컬럼**으로 쪼개고 해당하는 곳만 1.

> **왜 LabelEncoder 로 0,1,2 를 안 쓰나요?** 0,1,2 는 "2 가 0 보다 크다"는 **크기 관계**를 만들어 버려요. 계약 형태엔 크기가 없죠.
> 그래서 입력(X) 쪽 범주형은 원-핫이 안전해요. (공식 샘플 9번: object 컬럼 전부 원-핫 → `df_preset`)

### ✏️ 문제 13. object 컬럼 고르기
아직 문자열(object)인 컬럼 이름들을 **`obj_cols`** 에 저장하세요.

In [ ]:
obj_cols = df.___.columns
list(obj_cols)

<details><summary>💡 힌트 (막히면 클릭)</summary>

`select_dtypes('object')` → 문자열 컬럼만 / `select_dtypes('number')` → 숫자 컬럼만

</details>

In [ ]:
check.p03_q13()

### ✏️ 문제 14. get_dummies
`obj_cols` 를 원-핫 인코딩해 **`df_preset`** 에 저장하세요.

In [ ]:
df_preset = pd.___(data=df, columns=___)
print(df.shape, '→', df_preset.shape)
df_preset.head(3)

<details><summary>💡 힌트 (막히면 클릭)</summary>

`pd.get_dummies(data=df, columns=[...])`

</details>

In [ ]:
check.p03_q14()

> ⚠️ **함정**: 최신 pandas 의 `get_dummies` 는 0/1 대신 **True/False(bool)** 를 만들어요.
> 머신러닝(sklearn)은 괜찮지만, **딥러닝(PyTorch)** 텐서로 바꿀 때 에러가 날 수 있어요.
> 해결: `pd.get_dummies(..., dtype=int)` 또는 나중에 `X.astype('float32')`

In [ ]:
df_preset.dtypes.value_counts()   # bool 컬럼이 보이죠?

## 5. 학습용 / 검증용 나누기 `train_test_split`
**모의고사를 실제 시험 전에 미리 보면 안 되는 것**과 같아요. 모델이 공부한 데이터(train)와 시험 볼 데이터(test)를 나눠야 진짜 실력을 알 수 있어요.

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
```
- 반환 순서: **X_train, X_test, y_train, y_test** (외우기: X 두 개 먼저, y 두 개 나중)
- `random_state` : 매번 같은 방식으로 섞기 (문제에서 주는 숫자 그대로!)
- `stratify=y` : 분류 문제에서 train/test 의 정답 비율을 똑같이 유지

### ✏️ 문제 15. X / y 나누기
`df_preset` 에서 정답 `Churn` 을 뺀 나머지를 **`X`**, `Churn` 을 **`y`** 에 저장하세요.

In [ ]:
X = df_preset.___
y = df_preset___
X.shape, y.shape

<details><summary>💡 힌트 (막히면 클릭)</summary>

X = 문제지(입력), y = 답안(정답)

</details>

In [ ]:
check.p03_q15()

### ✏️ 문제 16. train_test_split
**test_size=0.2**, **random_state=42**, **stratify=y** 로 나눠서 `X_train, X_test, y_train, y_test` 에 저장하세요.

In [ ]:
from sklearn.model_selection import ___
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=___, random_state=___, stratify=___)
X_train.shape, X_test.shape

<details><summary>💡 힌트 (막히면 클릭)</summary>

시험에서는 검증용 이름을 `X_valid, y_valid` 로 주기도 해요. **문제에 적힌 이름 그대로!**

</details>

In [ ]:
check.p03_q16()

## 6. 스케일링 — 단위 맞추기
`tenure`(0~72) 와 `TotalCharges`(0~8000) 는 단위가 너무 달라요. 큰 숫자가 모델을 좌지우지하지 않게 범위를 맞춰요.

| 스케일러 | 결과 | 언제 |
|---|---|---|
| `MinMaxScaler` | 0 ~ 1 사이 | 가장 무난, 딥러닝에 자주 |
| `StandardScaler` | 평균 0, 표준편차 1 | 정규분포에 가까울 때 |

> ⚠️ **데이터 누수(leakage) 금지**: `fit` 은 **X_train 으로만!** test 는 `transform` 만.
> (시험 범위를 미리 보고 공부하면 안 되는 것과 같아요)
```python
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)   # 학습 데이터로 min/max 를 배우고 변환
X_test  = scaler.transform(X_test)        # 배운 min/max 로 변환만
```

### ✏️ 문제 17. MinMaxScaler
MinMaxScaler 를 **`scaler`** 로 만들고 **`X_train_s`**, **`X_test_s`** 를 만드세요.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = ___
X_train_s = scaler.___(X_train)
X_test_s = scaler.___(X_test)
X_train_s.min(), X_train_s.max()

<details><summary>💡 힌트 (막히면 클릭)</summary>

train: fit_transform / test: transform

</details>

In [ ]:
check.p03_q17()

### ✏️ 문제 18. StandardScaler
StandardScaler 를 **`std_scaler`** 로 만들고 **`X_train_z`**, **`X_test_z`** 를 만드세요.

In [ ]:
from sklearn.preprocessing import ___
std_scaler = StandardScaler()
X_train_z = std_scaler.___
X_test_z = std_scaler.___
X_train_z.mean().round(3), X_train_z.std().round(3)

<details><summary>💡 힌트 (막히면 클릭)</summary>

결과는 numpy 배열이에요 (컬럼 이름이 사라짐). 컬럼 이름이 필요하면 미리 `X.columns` 를 저장해 두세요.

</details>

In [ ]:
check.p03_q18()

## 7. 결과 저장

### ✏️ 문제 19. to_csv
`df_preset` 을 **`telco_preprocessed.csv`** 로 저장하세요 (인덱스 제외).

In [ ]:
df_preset.to_csv('telco_preprocessed.csv', ___)

<details><summary>💡 힌트 (막히면 클릭)</summary>

`index=False`

</details>

In [ ]:
check.p03_q19()

## 🏁 마무리

In [ ]:
check.progress()

### 전처리 순서 체크리스트 (시험장에서 그대로)
1. 불필요 컬럼 삭제 (ID, 시간 문자열 등) — `drop(axis=1)`
2. 이상치 제거 — `df[df['a'] < 기준]`
3. 결측치 — `isnull().sum()` 확인 → `dropna()` / `fillna()`
4. 문자열 숫자화 — `to_numeric`, `LabelEncoder`(타깃), `get_dummies`(입력)
5. `X`, `y` 분리 → `train_test_split`
6. 스케일링 — fit 은 train 만!

다음: **`04_머신러닝_실습.ipynb`**